In [3]:
import requests, time, hashlib, csv
from pathlib import Path

RG_ID = "d9103c72-3807-4378-9ce7-b6f3e8fdd547"   # OK Computer
ALBUM = "to_pimp_a_butterfly"
HEADERS = {"User-Agent": "AlbumCoverProject/0.1 (akinbobola71@email.com)"}

out = Path("covers") / ALBUM
out.mkdir(parents=True, exist_ok=True)

# 1. List every release (edition) in the album
releases, offset = [], 0
while True:
    r = requests.get("https://musicbrainz.org/ws/2/release",
                     params={"release-group": RG_ID, "limit": 100, "offset": offset, "fmt": "json"},
                     headers=HEADERS, timeout=30)
    r.raise_for_status()
    data = r.json()
    releases += data["releases"]
    offset += 100
    if offset >= data["release-count"]:
        break
    time.sleep(1.1)
print(len(releases), "releases found")

# 2. Download the front image for each release, skipping exact duplicates
seen, rows = {}, []
for rel in releases:
    rid = rel["id"]
    row = {"release_id": rid, "date": rel.get("date", ""), "country": rel.get("country", ""),
           "has_front": False, "file": "", "duplicate_of": ""}
    caa = rel.get("cover-art-archive")
    if caa is None or caa.get("front"):
        time.sleep(1.1)
        r = requests.get(f"https://coverartarchive.org/release/{rid}", headers=HEADERS, timeout=30)
        if r.status_code == 200:
            front = next((i for i in r.json()["images"] if i.get("front")), None)
            if front:
                url = front["thumbnails"].get("500") or front["image"]
                img = requests.get(url, headers=HEADERS, timeout=60).content
                h = hashlib.sha256(img).hexdigest()
                row["has_front"] = True
                if h in seen:
                    row["duplicate_of"] = seen[h]
                else:
                    (out / f"{rid}.jpg").write_bytes(img)
                    seen[h] = f"{rid}.jpg"
                    row["file"] = f"{rid}.jpg"
    rows.append(row)

# 3. Save a manifest and print the feasibility numbers
with open(out / "manifest.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)

print("Releases total:          ", len(rows))
print("Releases with front art: ", sum(r["has_front"] for r in rows))
print("Distinct images (exact): ", len(seen))

23 releases found
Releases total:           23
Releases with front art:  21
Distinct images (exact):  20
